In [16]:
import numpy as np
import matplotlib.pyplot as plt
from Utilities import extractor
import uproot
import awkward as ak    

x_MH175=extractor("Dati/Tprime_tAq_1800_MH175_LH_2017.root", "Events")


file=uproot.open("Dati/Tprime_tAq_1800_MH175_LH_2017.root")
tree=file["Events"]
booleans= tree.arrays(["FatJet_isMatchedWithA"], library="ak")
booleanas=tree.arrays(["FatJet_isMatchedWith2BHadrons"], library="ak")
Fatjet_isMatchedWithA= booleans["FatJet_isMatchedWithA"]
Fatjet_isMatchedWith2BHadrons= booleanas["FatJet_isMatchedWith2BHadrons"]

#Filtriamo i dati

mask = (ak.flatten(Fatjet_isMatchedWithA) == 1) & (ak.flatten(Fatjet_isMatchedWith2BHadrons) == 1)
x_filtered = x_MH175[mask]


In [17]:
from scipy.special import voigt_profile
from iminuit import Minuit
from iminuit.cost import LeastSquares

x_plot=list(x_filtered)
x_plot.sort()
x_easy=[x for x in x_plot if 125 < x < 225]

def voigt(x, norm, mu, sigma, gamma):
    return voigt_profile(x-mu, sigma, gamma) * norm

bin_counts, bin_edges = np.histogram(x_easy, bins=50)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]
bin_densities = bin_counts / (len(x_easy) * bin_width)  # Densità normalizzata
yerr=np.sqrt(bin_counts) / (len(x_easy) * bin_width) # Errore standard per i dati binned

ls_voigt=LeastSquares(bin_centers, bin_densities, yerr, model=voigt)

m_voigt=Minuit(ls_voigt,  norm=1, mu=175, sigma=5, gamma=1)
m_voigt.limits["mu"]= (150, 200)
m_voigt.limits["sigma"]= (0.1, 20)
m_voigt.limits["gamma"]= (0.01, 10)
m_voigt.migrad()


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 1032 (χ²/ndof = 22.4)      │              Nfcn = 146              │
│ EDM = 1.79e-06 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ norm  │   1.057   │   0.006   │            │            │         │         │       │
│ 1 │ mu    │  177.30   │   0.09    │            │            │   150   │   200   │       │
│ 2 │ sigma │   11.62   │   0.22    │            │            │   0.1   │   20    │       │
│ 3 │ gamma │   5.73    │   0.22    │            │            │  0.01   │   10    │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬─────────────────────────────────────┐
│       │     norm       mu    sigma    gamma │
├───────┼─────────────────────────────────────┤
│  norm │ 3.76e-05  0.02e-3 -0.60e-3  0.69e-3 │
│    mu │  0.02e-3  0.00842   -0.005    0.002 │
│ sigma │ -0.60e-3   -0.005   0.0475    -0.04 │
│ gamma │  0.69e-3    0.002    -0.04   0.0486 │
└───────┴─────────────────────────────────────┘

In [18]:
fit_MH175_values={}
fit_MH175_errors={}

fit_values={'MH175': fit_MH175_values,}
fit_errors={'MH175_errors': fit_MH175_errors}

for param in m_voigt.parameters:
    fit_MH175_values[param] = m_voigt.values[param]

for error in m_voigt.parameters:    #Qui non ho capito come fa a capire che deveestarre gli errori 
    fit_MH175_errors[error] = m_voigt.errors[error]

print(fit_MH175_values)
print(fit_MH175_errors)

import json
#QUi sono andato di metodo oragutang, ho deciso di voler fare 2 file separati peer errori e valori 
#Ho tenuto lo stesso quello con tutti i valori, casomai cambiassi idea

with open("fit_results.json", "r") as f:
    results=json.load(f)

with open("fit_values.json", "r") as g:
    values=json.load(g) 

with open("fit_errors.json", "r") as h:
    errors=json.load(h)


results["MH175"]=fit_MH175_values
results["MH175_errors"]=fit_MH175_errors

with open("fit_results.json", "w") as f:
    json.dump(results, f, indent=1)

values["MH175"]=fit_MH175_values
with open("fit_values.json", "w") as g:
    json.dump(values, g, indent=1)  

errors["MH175_errors"]=fit_MH175_errors
with open("fit_errors.json", "w") as h:
    json.dump(errors, h, indent=1)  


{'norm': 1.0573040274165961, 'mu': 177.30337336825957, 'sigma': 11.61699522957357, 'gamma': 5.734426541113037}
{'norm': 0.0061319114036518085, 'mu': 0.09177286773847015, 'sigma': 0.21795430750681533, 'gamma': 0.22031218002213482}
